#Requirements

In [1]:
from pathlib import Path
import argparse
import os
import random
import torch
import numpy as np
import pandas as pd
from datetime import datetime
import pickle
from torch.utils import data
import sys
import importlib
import glob

In [2]:
import os
from pathlib import Path

# Define the new directory and change the current working directory
new_directory = r'/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan'
os.chdir(new_directory)

# Verify the change
print("New Current Directory:", os.getcwd())

New Current Directory: /content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan


In [3]:
# Install the required packages
!pip install fair-esm
!pip install tape_proteins
!pip install transformers
!pip install rdkit-pypi
!pip install pypdb
!pip install tpot
!pip install tap
!pip install levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 13.2 MB/s eta 0:00:00
  Created

In [4]:
def modify_mc_bin_client():
    file_path = '/usr/local/lib/python3.10/dist-packages/mc_bin_client/mc_bin_client.py'

    # Check if the file exists
    if not os.path.exists(file_path):
        print(f"File {file_path} does not exist.")
        return

    # Read the file
    with open(file_path, 'r') as file:
        file_contents = file.read()

    # Replace the outdated exception handling syntax
    modified_contents = file_contents.replace('except MemcachedError, e:', 'except MemcachedError as e:')

    # Replace the exceptions.Exception
    modified_contents = modified_contents.replace('exceptions.Exception', 'Exception')


    # Remove the import of the non-existent 'exceptions' module
    modified_contents = modified_contents.replace('import exceptions', '')

    # Apply new modifications
    # Replace imports from memcacheConstants with relative imports
    modified_contents = modified_contents.replace('from memcacheConstants import', 'from .memcacheConstants import')

    if 'from . import memcacheConstants' not in modified_contents:
        modified_contents = modified_contents.replace('import memcacheConstants', 'from . import memcacheConstants')


    # Write the modified contents back to the file
    with open(file_path, 'w') as file:
        file.write(modified_contents)

    print(f"File {file_path} has been modified.")

def create_exceptions_file():
    import os

    # Define the directory and file path
    directory = '/usr/local/lib/python3.10/dist-packages/mc_bin_client'
    file_path = os.path.join(directory, 'exceptions.py')

    # Ensure the directory exists
    if not os.path.exists(directory):
        print("The directory does not exist. Please check the path.")
        return

    # Create and write to exceptions.py
    with open(file_path, 'w') as file:
        file.write('import builtins\n')

    print("exceptions.py file has been created successfully with 'import builtins'.")

In [5]:
# Execute the setup steps
modify_mc_bin_client()
create_exceptions_file()

File /usr/local/lib/python3.10/dist-packages/mc_bin_client/mc_bin_client.py has been modified.
exceptions.py file has been created successfully with 'import builtins'.


# Seq-to-func

## Getting .fasta File

In [ ]:
def Get_processed_data_tuples(sequences, max_seqs_len, save_fasta=False, output_file=None):
    # Ensure the input is a list of strings
    if not all(isinstance(seq, str) for seq in sequences):
        raise ValueError("All elements in the sequences list must be strings.")

    # Filter sequences that exceed the max length
    filtered_sequences = [
        (f"seq{i+1}", seq)
        for i, seq in enumerate(sequences)
        if len(seq) <= max_seqs_len
    ]

    # Save to a FASTA file if requested
    if save_fasta:
        with open(output_file, "w") as fasta_file:
            for seq_id, seq in filtered_sequences:
                fasta_file.write(f">{seq_id}\n{seq}\n")
        print("\n>>> Fasta file has been saved in: ", output_file)
    else:
        print("\n>>> save_flag is off for fasta file.")

    return filtered_sequences


## Getting ESM-1b embedding

In [ ]:
import esm
def generate_embeddings(data_set, model_select="ESM_1B", batch_size=64, save_flag= False, output_file=None):

    print("\n")

    if model_select == "ESM_1B":
        model, alphabet = esm.pretrained.esm1b_t33_650M_UR50S()
    else:
        raise ValueError("Only 'ESM_1B' model is supported in this implementation.")

    # Prepare the batch converter
    batch_converter = alphabet.get_batch_converter()

    # Set the model to evaluation mode and move it to GPU
    model.eval()
    model.cuda()


    chunk_size = 500  # Number of sequences per chunk
    batch_counter = 0  # File counter
    seq_ids_chunk = []  # To store sequence IDs for the current chunk
    seq_all_hiddens_chunk = []  # To store all hidden states for the current chunk
    seq_encodings_chunk = []  # To store sequence representations for the current chunk

    for i in range(0, len(data_set), batch_size):
        print(f"Processing batch {i} out of {len(data_set)}")
        if i + batch_size <= len(data_set):
            batch = data_set[i:i + batch_size]
        else:
            batch = data_set[i:]

        # Process the batch
        batch_labels, batch_strs, batch_tokens = batch_converter(batch)
        batch_tokens = batch_tokens.cuda()
        with torch.no_grad():
            results = model(batch_tokens, repr_layers=[33])
        results = results["representations"][33].cpu().detach()

        for j, (_, seq) in enumerate(batch):
            seq_all_hiddens_chunk.append(results[j, 1:len(seq) + 1].numpy())
            seq_encodings_chunk.append(results[j, 1:len(seq) + 1].mean(0).numpy())
            seq_ids_chunk.append(batch_labels[j])

        # Save chunk if the accumulated sequences exceed chunk_size
        if len(seq_ids_chunk) >= chunk_size or (i + batch_size) >= len(data_set):
            output_file_name = str(output_file).replace(".p", "")
            output_file_chunk = f"{output_file_name}_{batch_counter}.p"
            seq_embedding_output = {
                "seq_embeddings": np.array(seq_encodings_chunk),  # Save all chunk encodings
                "seq_ids": seq_ids_chunk,  # Save all chunk IDs
                "seq_all_hiddens": seq_all_hiddens_chunk,  # Save all chunk hidden states
            }
            with open(output_file_chunk, "wb") as f:
                pickle.dump(seq_embedding_output, f)
            print(f"Saved embeddings chunk to {output_file_chunk}")

            # Reset variables for the next chunk
            seq_ids_chunk = []
            seq_all_hiddens_chunk = []
            seq_encodings_chunk = []
            batch_counter += 1

    print("All batches processed and saved.")

## Prediction

### Check to retrieve the save model

In [ ]:
import Marjan_N05B_SQembSAtt_y as N05B
importlib.reload(N05B)
from Marjan_N05B_SQembSAtt_y import ATT_dataset
from Marjan_N05B_SQembSAtt_y import SQembSAtt_Model

Hiiiiiiiiiiiiiiiiii N05B
Hiiiiiiiiiiiiiiiiii N05B


In [ ]:
def load_model_file(model_name):
    model_path = ""
    if model_name == "sarkysian1k_ESM1B":
        model_path = "/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/Models_N05B/sarkysian_1ksampled/sarkysian_1ksampled_20240819_205947.pth"
    elif model_name == "sarkysian5k_ESM1B":
        model_path = "/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/Models_N05B/sarkisyan_5ksampled/sarkisyan_5ksampled_1B_20241105_160400.pth"
    elif model_name == "sarkysian10k_ESM1B":
        model_path = "/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/Models_N05B/sarkisyan_10ksampled/sarkisyan_10ksampled_1B_20241119_215807.pth"
    elif model_name == "sarkysianFull_ESM1b":
        model_path = "/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/Models_N05B/sarkisyan/sarkisyan_1B_20241218_060837.pth"
    elif model_name == "normalized_sarkysian1k_ESM1B":
        model_path = "/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/Models_N05B/normalized_sarkysian_1ksampled/normalized_sarkysian_1ksampled_20240822_173529.pth"
    elif model_name == "sarkisyan_Unirep":
        model_path = "/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/Models_N05B/sarkisyan_GFP/sarkisyan_GFP_Unirep_20240909_061220.pth"
    else:
        print("There is no model file with this name!!!!!!!!")

    return model_path

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
### set one of the types of the model using below list
### Note: name of model better to have only 2 parts
model_name_list = [ "sarkysian1k_ESM1B",  #0
                    "sarkysian5k_ESM1B",  #1
                    "sarkysian10k_ESM1B", #2
                    "sarkysianFull_ESM1b", #3
                    "normalized_sarkysian1k_ESM1B",  #4
                    "sarkisyan_Unirep"  #5
                  ]

model_name = model_name_list[3]
model_path = load_model_file(model_name)
model_params = torch.load(model_path)

# Extract hyperparameters
d_model = model_params['d_model']
d_k = model_params['d_k']
n_heads = model_params['n_heads']
d_v = model_params['d_v']
out_dim = model_params['out_dim']
d_ff = model_params['d_ff']
NN_type = model_params['NN_type']
log_value = model_params['log_value']
prpty_select = model_params['prpty_select']
data_folder = model_params['data_folder']
batch_size = model_params['batch_size']
learning_rate = model_params['learning_rate']
dataset_name = model_params['dataset_name']

# Print the variables
print("model_name: ", model_name)
print("d_model:", d_model)
print("d_k:", d_k)
print("n_heads:", n_heads)
print("d_v:", d_v)
print("out_dim:", out_dim)
print("d_ff:", d_ff)
print("NN_type:", NN_type)
print("log_value:", log_value)
print("prpty_select:", prpty_select)
print("data_folder:", data_folder)
print("batch_size:", batch_size)
print("learning_rate:", learning_rate)
print("dataset_name:", dataset_name)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<ipython-input-10-cc72e1c76961>:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted 

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
model = N05B.SQembSAtt_Model(d_model=d_model, d_k=d_k, n_heads=n_heads, d_v=d_v, out_dim=out_dim, d_ff=d_ff)

# Load the state dictionary into the model
model.load_state_dict(model_params['state_dict'])
model = model.double().cuda()
model.eval()

### Save the prediction

In [6]:
def save_prediciton_files(N05B, model, sequences, embedding_folder, embedding_file, prpty_select, batch_size=64, save_flag= False, output_file=None):

    X_seqs_all_hiddens_list, properties_dict = N05B.dataPrepFiles(embedding_folder, embedding_file = embedding_file, properties_file= None)


    seq_data = sequences

    X_seqs_all_hiddens = []
    aa_seqs = []

    print("len(X_seqs_all_hiddens_list):", len(X_seqs_all_hiddens_list))
    print("len(seq_data):", len(seq_data))

    for j in range(len(X_seqs_all_hiddens_list)):
        # Directly append all embeddings and sequences without filtering based on y
        X_seqs_all_hiddens.append(X_seqs_all_hiddens_list[j])
        aa_seqs.append(seq_data[j])

    loader = N05B.ATT_loader_no_y_values(N05B.ATT_dataset_no_y_values, aa_seqs, X_seqs_all_hiddens, batch_size)


    y_pred = []
    sequences = []  # Collect sequences here

    for one_seq_ppt_group in loader:
        # Get embeddings and masks from the batch
        seq_rep = one_seq_ppt_group["embedding"].float().cuda()
        seq_mask = one_seq_ppt_group["mask"].float().cuda()

        seq_rep, seq_mask = seq_rep.double().cuda(), seq_mask.double().cuda()

        # Forward pass through the model
        output, _ = model(seq_rep, seq_mask)

        # Collect predictions
        output = output.view(-1).cpu().detach().numpy()  # Flatten and move to CPU
        y_pred.extend(output)

        # Collect sequences
        batch_sequences = one_seq_ppt_group["sequences"]
        sequences.extend(batch_sequences)

    if len(sequences) == len(y_pred):
        # Create DataFrame including sequences and predictions
        result_df = pd.DataFrame({
            'sequences': sequences,
            prpty_select: y_pred  # Use prpty_select as the column name for predictions
        })
    else:
        print(f"len of sequences: {len(sequences)} and y_pred:{len(y_pred)} are not equal.....")

    if save_flag:
        result_df.to_csv(output_file, index=False)
        print(f"\n>>> Prediction File saved at: {output_file}")
    else:
        print("\n>>> Save_flas is off for prediction resuls.")
    return result_df

In [ ]:
# # Define the prediction folder
# prediction_folder = os.path.join(data_folder, "Prediction_results/fullModelPrediction")
# # full_model_folder = os.path.join(data_folder, "fullModel_files")
# os.makedirs(prediction_folder, exist_ok=True)
# model_select = "ESM_1B"
# name = "marjan"
# embedding_folder=None
# embedding_file = f"N03_{name}_embedding_{model_select}.p"
# X_seqs_all_hiddens_list, properties_dict = N05B.dataPrepFiles(embedding_folder, embedding_file = embedding_file, properties_file= None)



>>> Getting all input files and splitting the data... 
embedding file path:  /content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N03_marjan_embedding_ESM_1B.p
Sequence embeddings found in mulitple files.
>>> properties_dict is None.....


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
# import glob
# from pathlib import Path
# import pandas as pd



# files = {
#     "mcmc_lowN_all": Path("/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/N00_raw_datasets_processed/GFP/mcmc_GFP_uniqSeq.csv"),
#     "evodiff_avGFP_10ksamples": Path("/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/N00_raw_datasets_processed/GFP/evodiff_avGFP_10ksamples.csv"),
#     "evodiff_randedit_10ksamples": Path("/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/N00_raw_datasets_processed/GFP/evodiff_randedit_10ksamples.csv")
# }

# data_folder = "/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/mcmc"

# # Function to check if chunked embedding files exist
# def check_embedding_files_exist(base_path, name, model_select):
#     pattern = f"{base_path}/N03_{name}_embedding_{model_select}_*.p"
#     chunk_files = glob.glob(pattern)
#     return len(chunk_files) > 0, chunk_files

# max_seqs_len=1200
# save_flag = True
# model_select = "ESM_1B"

# for name, file_path in files.items():

#     # Read the data and extract the "SEQ" column
#     data = pd.read_csv(file_path)
#     sequences = data["SEQ"]

#     # Check for chunked embedding files
#     embedding_exists, embedding_files = check_embedding_files_exist(str(data_folder), name, model_select)
#     if not embedding_exists:
#         print(f"Generating embeddings for {name}...")
#         # embeddings = generate_embeddings(
#         #     processed_data_tuples,
#         #     model_select,
#         #     batch_size=batch_size,
#         #     save_flag=True,
#         #     output_file=data_folder / f"N03_{name}_embedding_{model_select}"
#         # )
#     else:
#         print(f"Embedding files for {name} already exist. Found {len(embedding_files)} chunk(s). Skipping generation.")


Embedding files for mcmc_lowN_all already exist. Found 5 chunk(s). Skipping generation.
Generating embeddings for evodiff_avGFP_10ksamples...
Generating embeddings for evodiff_randedit_10ksamples...


In [ ]:
import glob
# Function to check if chunked embedding files exist
def check_embedding_files_exist(base_path, name, model_select):
    pattern = f"{base_path}/N03_{name}_embedding_{model_select}_*.p"
    chunk_files = glob.glob(pattern)
    return len(chunk_files) > 0, chunk_files

In [ ]:
from pathlib import Path
import os
import pandas as pd

# Define the file paths and their names
files = {
    "mcmc_lowN_all": Path("/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/N00_raw_datasets_processed/GFP/mcmc_GFP_uniqSeq.csv"),
    "evodiff_avGFP_10ksamples": Path("/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/N00_raw_datasets_processed/GFP/evodiff_avGFP_10ksamples.csv"),
    "evodiff_randedit_10ksamples": Path("/content/drive/MyDrive/Colab Notebooks/Marjan/seq-to-func/0A - marjan/N_DataProcessing/N00_raw_datasets_processed/GFP/evodiff_randedit_10ksamples.csv")
}

max_seqs_len=1200
save_flag = True
model_select = "ESM_1B"


# Define the prediction folder
prediction_folder = os.path.join(data_folder, "Prediction_results/fullModelPrediction")
# full_model_folder = os.path.join(data_folder, "fullModel_files")
os.makedirs(prediction_folder, exist_ok=True)

# Loop through the files dictionary
for name, file_path in files.items():
    # Generate the file name for prediction output
    file_name = f"{model_name}_{name}.csv"
    prediction_file = os.path.join(prediction_folder, file_name)
    # Read the data and extract the "SEQ" column
    data = pd.read_csv(file_path)
    sequences = data["SEQ"]

    #example name: N00_final_org_sarkisyan.fasta
    fasta_file = data_folder / f"N00_{name}.fasta"
    processed_data_tuples = Get_processed_data_tuples(sequences, max_seqs_len, save_flag, fasta_file)

    #Example name: N03_mcmc_GFP_uniqSeq_1ksampled_embedding_Unirep.p
    #embedding_file = data_folder / f"N03_{name}_embedding_{model_select}.p"  # Assuming embeddings saved as .npy or any other format
    #embeddings = generate_embeddings(processed_data_tuples, model_select, batch_size=batch_size, save_flag=True, output_file=embedding_file)
    # Check for chunked embedding files
    embedding_exists, embedding_files = check_embedding_files_exist(str(data_folder), name, model_select)
    if not embedding_exists:
        print(f"Generating embeddings for {name}...")
        embeddings = generate_embeddings(
            processed_data_tuples,
            model_select,
            batch_size=batch_size,
            save_flag=True,
            output_file=data_folder / f"N03_{name}_embedding_{model_select}.p"
        )
    else:
        print(f"Embedding files for {name} already exist. Found {len(embedding_files)} chunk(s). Skipping generation.")

    # Perform prediction and save results
    df_result = save_prediciton_files(
        N05B, model, sequences, embedding_folder=data_folder,
        embedding_file=embedding_file,
        prpty_select=prpty_select, batch_size=batch_size,
        save_flag=True, output_file=prediction_file
    )
    print(f"Processed {name}: Results saved to {prediction_file}")
    print("======================================================")


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


NameError: name 'data_folder' is not defined